# 歌词采集-QQ音乐
不需要按专辑采集

In [1]:
import json
import pandas as pd

import os

In [2]:
import sys

sys.path.append('..')

from data_crawler import  get_songs_data_raw, get_all_songs_lyric, clear_and_save_lyric
from songs_libs import format_timestamp, SongDataCleaner

In [3]:
albums_to_delete = ['声生不息', '我歌', '中国梦', '谁是大歌神', '梦想的声音', '我是歌手', 'JJ的咖啡调调', '不凡的改变', '“17聚幸福”江苏卫视2017跨年演唱会', '江苏卫视', '湖南卫视', '浙江卫视', '启航2020', '2018中国蓝', '时光音乐会', '经典咏流传', '天籁', '剧好听的歌']

# 批量数据采集

In [ ]:
singers = [('luodayou', '罗大佑'), ('lizongsheng', '李宗盛'), ('zhangxueyou', '张学友'), ('twins', 'Twins'), ('wangsulong', '汪苏泷'), ('panweibo', '潘玮柏'), ('dengziqi', 'G.E.M. 邓紫棋'), ('xuezhiqian', '薛之谦'), ('xusong', '许嵩'), ('zhangjie', '张杰'), ('taozhe', '陶喆'), ('fangdatong', '方大同'), ('wangfei', '王菲'), ('maobuyi', '毛不易'), ('beyond', 'BEYOND')]
max_page = 10

for i in singers[-1:]:
    file_path_prefix = f"data/{i[0]}/"
    # 如果file_path_prefix不存在，则新建
    if not os.path.exists(file_path_prefix):
        os.makedirs(file_path_prefix)
    # 原始曲目数据采集
    song_data_raw = get_songs_data_raw(singger=i[0], max_page=max_page)
    df_song_data_raw = pd.DataFrame(song_data_raw)
    # 原始曲目保存
    df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)
    # 重新读取数据
    df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
    song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')
    # 删除live歌曲
    df_songs = SongDataCleaner.clear_live_songs(df_song_data_raw_read)
    # 歌曲名清洗
    df_songs = SongDataCleaner.clear_song_name(df_songs)
    # 仅含歌手独唱歌曲
    df_songs = SongDataCleaner.clear_song_singer(df_songs, i[1])
    # 删除晚会歌曲
    df_songs = SongDataCleaner.clear_song_tv_show(df_songs, albums_to_delete)
    df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)

    df_songs_final = df_songs.head(130)
    df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
        lambda x: format_timestamp(x))
    df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
    df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
        lambda x: x.split('-')[0])
    df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)
    # 歌词采集与清洗
    get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)
    clear_and_save_lyric(file_path_prefix, df_songs_final)

# main

In [4]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

# file_path_prefix = "data/liyuchun/"
# singer = "李宇春"
# max_page = 14

file_path_prefix = "data/maobuyi/"
singer = "毛不易"
max_page = 20
if not os.path.exists(file_path_prefix):
    os.makedirs(file_path_prefix)

### 曲目采集

In [ ]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singer, max_page=max_page)

In [ ]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [5]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')
df_song_data_raw_read

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time
0,251875009,001Cnurq0Oe2cM,一程山路,NaN,毛不易,1507534,001BHDR33FZVZ0,小王,9495273,001OFJ154OfZuW,215,1582128000
1,336582682,002QhULf16tWw1,无名的人,《雄狮少年》电影主题曲,毛不易,1507534,001BHDR33FZVZ0,无名的人,24100646,002mPgSG01LLtu,282,1639411200
2,203451421,003kLvu04bLGzi,消愁 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,明日之子 第7期,2188465,002xoonH2Bk7FR,179,1501257600
3,203514624,00375L600p9sxv,像我这样的人 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,明日之子 第8期,2196371,0001n7a82gh6IY,171,1501862400
4,254554296,002XkEH930NXSr,一荤一素 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,歌手·当打之年 第2期,10635028,002VxplL2gXAuH,314,1581609600
...,...,...,...,...,...,...,...,...,...,...,...,...
595,411157630,001JxoY21u6dTE,毛不易-杭州-3.25-前排饭拍-《东北民谣》,NaN,爱看演出的小ye,0,0032fmHO2UDnV3,NaN,0,NaN,136,0
596,410965372,000ltWTM14ERc5,毛不易-杭州-3.25-前排饭拍-《深夜一角》,NaN,爱看演出的小ye,0,0032fmHO2UDnV3,NaN,0,NaN,102,0
597,324524814,003pkuZ83alTIv,小王 TME live 毛不易2020「像我这样的人」全原创线上演唱会,NaN,毛不易;TME live,5921582,004N1O3F2QKvfi,NaN,0,NaN,342,1608235425
598,578756551,003snbyn0uJFUG,黑月光 (DJ阿lo版),NaN,张碧晨;毛不易,0,0032fmHO2UDnV3,NaN,0,NaN,193,0


### 清洗

In [6]:
# 删除live歌曲
# df_songs = SongDataCleaner.clear_live_songs(df_song_data_raw_read)
df_songs = df_song_data_raw_read[~df_song_data_raw_read['song_name'].str.
                                 contains('口白')]
df_songs = df_songs[~df_songs['song_name'].str.contains('现场版', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('&', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('【', case=False)]
df_songs = df_songs[~df_songs['song_name'].str.contains('／', case=False)]

# 歌曲名清洗
df_songs = SongDataCleaner.clear_song_name(df_songs)
# 仅含歌手独唱歌曲
df_songs = SongDataCleaner.clear_song_singer(df_songs, singer)
# 删除晚会歌曲
df_songs = SongDataCleaner.clear_song_tv_show(df_songs, albums_to_delete)
df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure
0,251875009,001Cnurq0Oe2cM,一程山路,NaN,毛不易,1507534,001BHDR33FZVZ0,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王
1,336582682,002QhULf16tWw1,无名的人,《雄狮少年》电影主题曲,毛不易,1507534,001BHDR33FZVZ0,无名的人,24100646,002mPgSG01LLtu,282,1639411200,无名的人,无名的人,无名的人
2,203451421,003kLvu04bLGzi,消愁 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,明日之子 第7期,2188465,002xoonH2Bk7FR,179,1501257600,消愁,消愁,明日之子 第7期
3,203514624,00375L600p9sxv,像我这样的人 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,明日之子 第8期,2196371,0001n7a82gh6IY,171,1501862400,像我这样的人,像我这样的人,明日之子 第8期
4,254554296,002XkEH930NXSr,一荤一素 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,歌手·当打之年 第2期,10635028,002VxplL2gXAuH,314,1581609600,一荤一素,一荤一素,歌手·当打之年 第2期
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,472529295,001wII8B4KmmjP,家乡人,NaN,毛不易,0,0032fmHO2UDnV3,NaN,0,NaN,382,0,家乡人,家乡人,nan
173,323323270,0041CPzR2CKskO,好物会发声,NaN,毛不易,0,0032fmHO2UDnV3,NaN,0,NaN,295,0,好物会发声,好物会发声,nan
174,349967702,002GUUvO3DDAl7,守岁 (Live),NaN,毛不易,0,0032fmHO2UDnV3,NaN,0,NaN,219,1643558400,守岁,守岁,nan
175,323607486,0028psfV46cNqY,4号病房,NaN,毛不易,0,0032fmHO2UDnV3,NaN,0,NaN,223,1514044800,4号病房,4号病房,nan


In [ ]:
# 特别处理
df_songs['album_name'] = df_songs['album_name'].fillna('').astype(str)
df_songs = df_songs[~df_songs['album_name'].str.contains('回蔚')]
df_songs

In [7]:
# 前130首
songs_list_all = df_songs['song_name_pure'].to_list()
songs_list_130 = df_songs.head(130)['song_name_pure'].to_list()

## 歌单确认

In [8]:
# 李宇春
songs_to_add = [
    '皇后与梦想', '下雨', '冰菊物语', '我的王国', '漂浮地铁', '今天有朵云爱我', '您所拨打的电话号码是空号',
    '一而再再而三地喜欢你', '人间乐园', 'TMD我爱你', '口音', '木兰', '开放'
]
songs_to_delete = ['今夜你会不会来', '春风十里', '情书', '那女孩对我说', '南方姑娘', '爱你所爱', '无心睡眠', '莫过于此', '天黑黑', '张三的歌', '漂洋过海来看你', '城里的月光', '不要对他说', '下个,路口,见']
# 陈奕迅
songs_to_delete = ['新曲+精选', 'K歌之王AIR', '慢慢喜欢你', '最冷一天']
# 任贤齐
songs_to_delete = ['伤心太平洋+心太软+我是一只鱼+对面的女孩看过来', '桥边姑娘', '你知道我在等你吗', '我是一只小小鸟', '外婆的澎湖湾2015', '海阔天空', '爱的路上只有你和我', '爱的初体验', '美丽的坏女人', '朋友的酒']
# 林俊杰
# songs_to_delete = ['开场白', '无聊', '起风了']
# 陈信宏
# songs_to_delete = ['玫瑰少年-FromTHEFIRSTTAKE', '疯狂世界+候鸟', '2010离开地球表面', '盛夏光年×HIPHOPMAN', 'Paradise+倔强', 'Hosee', 'SHERO']
# 蔡依林
# songs_to_delete = ['看我七十二变', '爱情36计']
# 伍佰
# songs_to_delete = ['少年吔,安啦！']
# 凤凰传奇
songs_to_delete = [
    '海底', 'mrs.leta', '好运来', '普通disco', '【拜新年】专辑歌曲串烧', '好汉歌', '狼的诱惑广场舞版',
    '专辑歌曲串烧', '歌曲串烧', 'dj天籁传奇', '天蓝蓝)'
]
# Beyond
songs_to_delete = [
    '为了你,为了我', '真的爱妳', '遥かなる夢に', '灰色軌跡', '遥かなる梦に〜Faraway〜',
    'リゾ·ラバ～International～', 'Cryin', '遥かなるゆめに～Faraway', '喜欢妳'
]

# 罗大佑
songs_to_delete = ['童年full']

# 方大同
songs_to_delete = ['假行僧', '金砖的秘密', '月亮代表我的心']

# 李宗盛
# songs_to_delete = ['我听见有人叫你宝贝', '17岁女生的温柔', '漂洋过海来看你']

# 莫文蔚
# songs_to_delete = ['当你老了', '夜上海']

# 毛不易
songs_to_add = ['消愁']
songs_to_delete = ['小王日记']



songs_list_final = songs_list_130.copy()

In [ ]:
# 添加
for i in songs_to_add:
    if i not in songs_list_final[:100]:
        print(i)
        # 添加到列表的第一个
        songs_list_final.insert(0, i)

In [9]:
# 删除
for i in songs_to_delete:
    if i in songs_list_final:
        print(i)
        songs_list_final.remove(i)

小王日记


In [10]:
len(songs_list_final)

129

## 歌曲数据确认

In [11]:
df_songs_final = df_songs[df_songs['song_name_pure'].isin(
    songs_list_final)].reset_index(drop=True)

# df_songs_final = df_songs_final.drop(columns=['song_name_unique'])
# df_songs_final['song_name_unique'] = df_songs_final['song_name_pure']
df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
    lambda x: format_timestamp(x))
df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
    lambda x: x.split('-')[0])
df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,251875009,001Cnurq0Oe2cM,一程山路,NaN,毛不易,1507534,001BHDR33FZVZ0,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
1,336582682,002QhULf16tWw1,无名的人,《雄狮少年》电影主题曲,毛不易,1507534,001BHDR33FZVZ0,无名的人,24100646,002mPgSG01LLtu,282,1639411200,无名的人,无名的人,无名的人,2021-12-14,2021
2,203451421,003kLvu04bLGzi,消愁 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,明日之子 第7期,2188465,002xoonH2Bk7FR,179,1501257600,消愁,消愁,明日之子 第7期,2017-07-29,2017
3,203514624,00375L600p9sxv,像我这样的人 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,明日之子 第8期,2196371,0001n7a82gh6IY,171,1501862400,像我这样的人,像我这样的人,明日之子 第8期,2017-08-05,2017
4,254554296,002XkEH930NXSr,一荤一素 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,歌手·当打之年 第2期,10635028,002VxplL2gXAuH,314,1581609600,一荤一素,一荤一素,歌手·当打之年 第2期,2020-02-14,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,503421737,0033mKxH3ELFKK,有你，就有新回忆,假日酒店Holiday Inn品牌主题曲,毛不易,1507534,001BHDR33FZVZ0,有你，就有新回忆 (假日酒店Holiday Inn品牌主题曲),53458324,000LvMNT3h6yVa,222,1713974400,有你就有新回忆,有你就有新回忆,有你，就有新回忆 (假日酒店Holiday Inn品牌主题曲),2024-04-25,2024
125,503409738,003VEsBQ1E8cdm,世间美好与你环环相扣 (2022壬寅年中央广播电视总台元宵晚会现场),NaN,毛不易,0,0032fmHO2UDnV3,NaN,0,NaN,141,1644854400,世间美好与你环环相扣,世间美好与你环环相扣,nan,2022-02-15,2022
126,345070442,001tbY85093M7Q,易燃易爆炸,NaN,毛不易,1507534,001BHDR33FZVZ0,NaN,0,NaN,201,1483200000,易燃易爆炸,易燃易爆炸,nan,2017-01-01,2017
127,292111304,0004EcHX2CB6GT,得过且过的勇者 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,2020最美的夜bilibili晚会,16601204,0002VXnS2aXARC,196,1609344000,得过且过的勇者,得过且过的勇者,2020最美的夜bilibili晚会,2020-12-31,2020


In [12]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

## 歌词采集

In [29]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)

感觉自己是巨星
或者我拥抱你
一个关于未来的想象
从无到有
Intro
青春当歌


In [26]:
# 读取raw_lyric_data.json
with open(file_path_prefix + 'raw_lyric_data.json', 'r', encoding='utf-8') as f:
    lyric_raw = json.load(f)
# 删除lyric_raw中lyric_raw为空的元素
lyric_raw = [song for song in lyric_raw if song['lyric_raw']]

# 替换raw_lyric_data.json
with open(file_path_prefix + 'raw_lyric_data.json', 'w', encoding='utf-8') as f:
    json.dump(lyric_raw, f, ensure_ascii=False, indent=4)

len(lyric_raw)

# 重新运行上一个cell

123

## 歌词清洗

In [30]:
clear_and_save_lyric(file_path_prefix, df_songs_final)

In [31]:
# 歌词数据查验
df_lyric = pd.read_json(f"{file_path_prefix}cleared_lyric_data.json")
df_lyric['lyric_length'] = df_lyric['lyrics_text'].apply(lambda x: len(x))
df_lyric.sort_values(by='lyric_length')

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text,lyric_length
116,638027906,Intro,NaT,0,,,,,0
101,426961144,一个关于未来的想象,NaT,0,,,,,0
75,251875006,Outro,2026-03-26 00:00:38,1,,,,如果我在角落里遇见他。碰巧有风吹乱他的头发。我会慢慢靠近给他肩膀。分担他一路重重的绝望。如果...,99
128,212809455,说散就散,2026-03-26 00:19:16,1,张楚翘,伍乐城,伍乐城,抱一抱。就当作从没有在一起。好不好。要解释都已经来不及。算了吧。我付出过什么没关系。我忽略自...,151
56,203813292,故乡游 (Live),2026-03-26 00:17:42,1,包珍妮,钟易轩,黄竣琮 by T.Y.Z,乡间的歌谣。仍是儿时哼的调。树上燕子筑鸟巢。街角卖过的小笼包。如今再也买不到。爷爷的草帽。不...,167
...,...,...,...,...,...,...,...,...,...
15,265408076,入海,2026-03-26 00:16:55,1,马晓波,赵兆,宋涛/赵兆,还有说不完的话。风催着我们出发。把笑和泪都留下。留在这一年的夏。对于未来的想法。有太多疑问没...,563
120,256565237,渴不停,2026-03-26 00:18:31,1,王韵韵,Henrik Tala/Jay Hong/Erik Smaaland/Alexander P...,Mike Spencer,瞳孔喷出一团火。唾液回流绕着舌。太阳晒得我好渴，燥热。体内热血沸腾。已到表皮真层。时速以秒计...,564
38,569424076,你没说的话,2026-03-26 00:16:10,1,王海涛,彭飞,张宗炜,那些你没说的话。其实我都懂。它从你心里，响到我耳旁。慌张无措的时候。你总拿笑来遮掩。这时我，...,584
104,221856451,远方的风 (Live),2026-03-26 00:32:24,1,完玛三智,完玛三智,黄竣琮@TYZ,远方的风。摇曳无常的花。谁结的果。抚慰流浪的马。远方的风。送我去远方的远方。青马山间。路人独...,629


In [32]:
# 歌词文本长度大于40
df_lyric = df_lyric[df_lyric['lyric_length'] > 100]
df_lyric

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text,lyric_length
0,251875009,一程山路,2026-03-26 00:28:00,1,,,,青石板留着谁的梦啊。一场秋雨，又落一地花。旅人匆匆地赶路啊。走四季，访人家。如同昨夜天光乍破...,241
1,336582682,无名的人,2026-03-26 00:17:00,1,唐恬 TIAN TANG,钱雷 LEI QIAN,钱雷 LEI QIAN,我是这路上，没名字的人。我没有新闻，没有人评论。要拼尽所有，换得普通的剧本。曲折辗转，不过谋...,503
2,203451421,消愁 (Live),2026-03-26 00:18:05,1,毛不易,毛不易,郑楠,当你走进这欢乐场。背上所有的梦与想。各色的脸上各色的妆。没人记得你的模样。三巡酒过你在角落。...,289
3,203514624,像我这样的人 (Live),2026-03-26 00:16:02,1,毛不易,毛不易,郑楠,像我这样优秀的人。本该灿烂过一生。怎么二十多年到头来。还在人海里浮沉。像我这样聪明的人。早就...,266
4,254554296,一荤一素 (Live),2026-03-26 00:27:32,1,毛不易,毛不易,赵兆,日出又日落。深处再深处。一张小方桌。有一荤一素。一个身影从容地忙忙碌碌。一双手让这时光有了温...,317
...,...,...,...,...,...,...,...,...,...
124,503421737,有你，就有新回忆,2026-03-26 00:19:50,1,刘兆伦,刘兆伦,弋洋,早安。枕头柔软触感，阳光铺陈温暖。第一声的问候，也用微笑交换。你看。孩子有点贪玩，围着沙发打...,363
125,503409738,世间美好与你环环相扣 (2022壬寅年中央广播电视总台元宵晚会现场),2026-03-26 00:20:01,1,尹初七,柏松,,偏偏秉烛夜游。午夜星辰，似奔走之友。爱你每个结痂伤口。酿成的陈年烈酒。入喉尚算可口。怎么泪水...,334
126,345070442,易燃易爆炸,2026-03-26 00:16:47,1,尚梦迪/骈然,陈粒,,盼我疯魔还盼我孑孓不独活。想我冷艳还想我轻佻又下贱。要我阳光还要我风情不摇晃。戏我哭笑无主还...,362
127,292111304,得过且过的勇者 (Live),2026-03-26 00:22:12,1,ilem,ilem,,勇者打着酒嗝离开酒馆。今天也保护村庄安全。不缺乏力量或者是正义感。但是我有点懒。听说西边出现...,380


In [ ]:
# 删除有问题的
song_ids_to_delete = [251875006]
df_lyric = df_lyric[~df_lyric['song_id'].isin(song_ids_to_delete)]
df_lyric

In [33]:
# 覆盖原文件
df_lyric['start_time'] = df_lyric['start_time'].astype(str).apply(lambda x: x.split(' ')[1])
df_lyric_d = df_lyric.to_dict(orient='records')
with open(file_path_prefix + 'cleared_lyric_data.json', 'w',
              encoding='utf-8') as f:
        json.dump(df_lyric_d, f, ensure_ascii=False, indent=4)

## 特别处理

In [ ]:
df_lyric['lyricist'].unique()

In [ ]:
lyricist = ['阿信', '五月天阿信', '五月天 阿信', '阿信(五月天)']
df_lyric_flited = df_lyric[df_lyric['lyricist'].isin(lyricist)]
df_lyric_flited

In [ ]:
songs_to_delete_lyric = ['派对动物 + 离开地球表面 (live in the sky)', '伤心的人就听撑腰 (Life Live)', '明白 (后段) (Live)']
df_lyric_flited = df_lyric_flited[~df_lyric_flited['song_name'].isin(songs_to_delete_lyric)]
df_lyric_flited

In [ ]:
# 将df_lyric_flited保存为json
df_lyric_flited = df_lyric_flited.copy()
df_lyric_flited['start_time'] = df_lyric_flited['start_time'].astype(str).apply(lambda x: x.split(' ')[1])
df_lyric_flited_d = df_lyric_flited.to_dict(orient='records')
with open(file_path_prefix + 'cleared_lyric_data.json', 'w',
              encoding='utf-8') as f:
        json.dump(df_lyric_flited_d, f, ensure_ascii=False, indent=4)